### <center><h1>Hetrogeneous Graph Attention Network (<em>HAN</em>) for TCGA-BRCA</h1></center></br>

In [ ]:
# Global constants for reproducibility
MASTER_RANDOM_SEED = 43  # Anyone using this seed will get identical results
EXPERIMENT_OFFSET = 1000  # Large offset to ensure different seeds per experiment

import time
import numpy as np
import tensorflow.compat.v1 as tf
import scipy.sparse as sp
from torch_geometric.data import HeteroData
from models import GAT, HeteGAT, HeteGAT_multi
from utils import process
from itertools import combinations, permutations
import json
import logging
from datetime import datetime
import os
from typing import List, Tuple, Dict, Any
import pandas as pd
import random
import sys
import warnings

# Comprehensive output suppression
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # Suppress all TensorFlow messages except errors
warnings.filterwarnings('ignore')  # Suppress all warnings

# Suppress TensorFlow logging
tf.logging.set_verbosity(tf.logging.ERROR)  # Only show errors from TensorFlow

# Set all random seeds at module level for complete reproducibility
def set_all_seeds(seed=MASTER_RANDOM_SEED):
    """Set all random seeds for complete reproducibility"""
    random.seed(seed)
    np.random.seed(seed)
    tf.set_random_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    # For TensorFlow deterministic operations - but less strict
    # os.environ['TF_DETERMINISTIC_OPS'] = '1'
    # os.environ['TF_CUDNN_DETERMINISTIC'] = '1'

# Initialize all seeds
set_all_seeds()

# Configure comprehensive logging
class MetaPathLogger:
    def __init__(self, log_dir="metapath_logs"):
        self.log_dir = log_dir
        os.makedirs(log_dir, exist_ok=True)
        
        # Setup main logger with unique name to avoid conflicts
        logger_name = f'MetaPathExperiment_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
        self.logger = logging.getLogger(logger_name)
        self.logger.setLevel(logging.INFO)
        
        # Clear any existing handlers to prevent duplicates
        self.logger.handlers.clear()
        
        # Prevent propagation to root logger to avoid duplicate messages
        self.logger.propagate = False
        
        # Create formatters
        detailed_formatter = logging.Formatter(
            '%(asctime)s - %(name)s - %(levelname)s - %(message)s'
        )
        
        # File handler for detailed logs
        file_handler = logging.FileHandler(
            os.path.join(log_dir, f'metapath_experiment_{datetime.now().strftime("%Y%m%d_%H%M%S")}.log')
        )
        file_handler.setLevel(logging.DEBUG)
        file_handler.setFormatter(detailed_formatter)
        
        # Console handler
        console_handler = logging.StreamHandler()
        console_handler.setLevel(logging.INFO)
        console_handler.setFormatter(detailed_formatter)
        
        self.logger.addHandler(file_handler)
        self.logger.addHandler(console_handler)
        
        # Results tracking
        self.results = []
        self.best_metapath = None
        self.best_accuracy = 0.0
        
    def log_experiment_start(self, hetero_data):
        """Log experiment initialization details"""
        print("=" * 80)
        print("STARTING ENHANCED DYNAMIC META-PATH EXPERIMENT")
        print("=" * 80)
        
        # Log graph structure
        node_types = list(hetero_data.node_types)
        edge_types = list(hetero_data.edge_types)
        
        print("\nGraph Structure:")
        print(f"  Node types: {node_types}\n")
        print(f"  Edge types: {edge_types}")
        
        # Log node counts
        print("\nNode Count:")
        for node_type in node_types:
            count = hetero_data[node_type].num_nodes
            print(f"  {node_type}: {count} nodes")

        # Log edge counts
        print("\nEdge Count:")
        for edge_type in edge_types:
            count = hetero_data[edge_type].num_edges
            print(f"  {edge_type}: {count} edges")            
    
    def log_metapath_generation(self, total_metapaths, by_category):
        """Log meta-path generation statistics"""
        print("\nMeta-path Generation Complete:")
        print(f"  \nTotal meta-paths generated: {total_metapaths}")
        for category, count in by_category.items():
            print(f"  {category}: {count} paths")
        
    def log_metapath_set_start(self, metapath_idx, metapath_set):
        """Log the start of training with a specific meta-path set"""
        print(f"\n{'='*60}")
        print(f"TRAINING META-PATH SET {metapath_idx + 1}")
        print(f"{'='*60}\n")
        
        for i, metapath in enumerate(metapath_set):
            path_str = " -> ".join([f"{src}--{rel}-->{dst}" for src, rel, dst in metapath])
            print(f"  Path {i+1}: {path_str}")
    
    def log_training_progress(self, epoch, train_loss, train_acc, val_loss, val_acc):
        """Log training progress"""
        if epoch % 10 == 0:  # Log every 10 epochs
            self.logger.debug(f"Epoch {epoch:3d}: "
                            f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
                            f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
    
    def log_metapath_result(self, metapath_set, train_acc, val_acc, test_acc, training_time):
        """Log results for a specific meta-path set"""
        result = {
            'metapath_set': metapath_set,
            'train_accuracy': float(train_acc),
            'val_accuracy': float(val_acc),
            'test_accuracy': float(test_acc),
            'training_time': training_time,
            'timestamp': datetime.now().isoformat()
        }
        
        self.results.append(result)
        
        # Update best meta-path
        if test_acc > self.best_accuracy:
            self.best_accuracy = test_acc
            self.best_metapath = metapath_set
            
        print("\nResults:")
        print(f"  Train Accuracy: {train_acc:.4f}")
        print(f"  Val Accuracy: {val_acc:.4f}")
        print(f"  Test Accuracy: {test_acc:.4f}")
        print(f"  Training Time: {training_time:.2f}s")
        
        if test_acc > self.best_accuracy - 0.001:  # Close to best
            print("\n  *** NEW BEST ACCURACY! ***")
    
    def log_final_summary(self):
        """Log final experiment summary"""
        print("\n" + "="*80)
        print("EXPERIMENT COMPLETE - FINAL SUMMARY")
        print("="*80)
        
        if self.results:
            # Sort results by test accuracy
            sorted_results = sorted(self.results, key=lambda x: x['test_accuracy'], reverse=True)
            
            print(f"\nTotal meta-path sets tested: {len(self.results)}")
            print(f"\nBest test accuracy: {self.best_accuracy:.4f}")
            
            print("\nTop 10 Meta-path Sets:")
            for i, result in enumerate(sorted_results[:10]):
                print(f"  {i+1}. Test Acc: {result['test_accuracy']:.4f}")
                for j, metapath in enumerate(result['metapath_set']):
                    path_str = " -> ".join([f"{src}--{rel}-->{dst}" for src, rel, dst in metapath])
                    print(f"     Path {j+1}: {path_str}")
            
            # Save results to JSON and CSV
            self.save_results()
        
        print("="*80)
    
    def save_results(self):
        """Save all results to files"""
        # Save to JSON
        json_path = os.path.join(self.log_dir, 'metapath_results.json')
        with open(json_path, 'w') as f:
            json.dump(self.results, f, indent=2)
        
        # Save to CSV for easy analysis
        csv_path = os.path.join(self.log_dir, 'metapath_results.csv')
        df_results = []
        for i, result in enumerate(self.results):
            row = {
                'rank': i + 1,
                'test_accuracy': result['test_accuracy'],
                'val_accuracy': result['val_accuracy'],
                'train_accuracy': result['train_accuracy'],
                'training_time': result['training_time'],
                'num_metapaths': len(result['metapath_set']),
                'metapath_description': self._metapath_set_to_string(result['metapath_set'])
            }
            df_results.append(row)
        
        # Sort by test accuracy for ranking
        df_results.sort(key=lambda x: x['test_accuracy'], reverse=True)
        for i, row in enumerate(df_results):
            row['rank'] = i + 1
        
        df = pd.DataFrame(df_results)
        df.to_csv(csv_path, index=False)
        
        print(f"Results saved to {json_path} and {csv_path}")
    
    def _metapath_set_to_string(self, metapath_set):
        """Convert meta-path set to readable string"""
        paths = []
        for metapath in metapath_set:
            path_str = " -> ".join([f"{src}--{rel}-->{dst}" for src, rel, dst in metapath])
            paths.append(path_str)
        return " | ".join(paths)

class EnhancedMetaPathGenerator:
    def __init__(self, hetero_data, target_node_type, max_path_length):
        self.hetero_data = hetero_data
        self.target_node_type = target_node_type
        self.max_path_length = max_path_length
        self.node_types = list(hetero_data.node_types)
        self.edge_types = list(hetero_data.edge_types)
        
        # Build comprehensive adjacency info
        self.adjacency_map = self._build_adjacency_map()
        self.reverse_adjacency_map = self._build_reverse_adjacency_map()
        
        # Categorize edge types for better path generation
        self.edge_categories = self._categorize_edges()
        
    def _build_adjacency_map(self):
        """Build a map of node type connections (forward direction)"""
        adj_map = {}
        for src, rel, dst in self.edge_types:
            if src not in adj_map:
                adj_map[src] = []
            adj_map[src].append((rel, dst))
        return adj_map
    
    def _build_reverse_adjacency_map(self):
        """Build a map of node type connections (reverse direction)"""
        rev_adj_map = {}
        for src, rel, dst in self.edge_types:
            if dst not in rev_adj_map:
                rev_adj_map[dst] = []
            rev_adj_map[dst].append((f"rev_{rel}", src))
        return rev_adj_map
    
    def _categorize_edges(self):
        """Categorize edges by their semantic meaning"""
        categories = {
            'direct_case_relations': [],
            'case_to_molecular': [],
            'molecular_to_case': [],
            'molecular_interactions': [],
            'subtype_relations': [],
            'case_similarity': [],
            'molecular_similarity': []
        }
        
        for src, rel, dst in self.edge_types:
            if src == 'case' and dst == 'subtype':
                categories['subtype_relations'].append((src, rel, dst))
            elif src == 'subtype':
                categories['subtype_relations'].append((src, rel, dst))
            elif src == 'case' and dst == 'case':
                categories['case_similarity'].append((src, rel, dst))
            elif src == 'case' and dst in ['gene', 'protein', 'cnv', 'mutation']:
                categories['case_to_molecular'].append((src, rel, dst))
            elif src in ['gene', 'protein', 'cnv', 'mutation'] and dst == 'case':
                categories['molecular_to_case'].append((src, rel, dst))
            elif src in ['gene', 'protein', 'cnv', 'mutation'] and dst in ['gene', 'protein', 'cnv', 'mutation']:
                if src == dst:
                    categories['molecular_similarity'].append((src, rel, dst))
                else:
                    categories['molecular_interactions'].append((src, rel, dst))
        
        return categories
    
    def generate_comprehensive_metapaths(self, max_paths_per_length):
        """Generate comprehensive meta-paths including all possible relations"""
        all_metapaths = []
        
        # 1. Direct case-to-case paths (similarity)
        direct_paths = self._generate_direct_case_paths()
        all_metapaths.extend(direct_paths)
        
        # 2. Case-subtype-case paths
        subtype_paths = self._generate_subtype_mediated_paths()
        all_metapaths.extend(subtype_paths)
        
        # 3. Case-molecular-case paths (through genes, proteins, etc.)
        molecular_paths = self._generate_molecular_mediated_paths()
        all_metapaths.extend(molecular_paths)
        
        # 4. NEW: Comprehensive multi-hop paths using DFS exploration
        complex_paths = self._generate_comprehensive_multihop_paths(max_paths_per_length)
        all_metapaths.extend(complex_paths)
        
        # 5. Hybrid paths combining different types of relations
        hybrid_paths = self._generate_hybrid_paths()
        all_metapaths.extend(hybrid_paths)
        
        return all_metapaths
    
    def _generate_direct_case_paths(self):
        """Generate direct case-to-case similarity paths"""
        paths = []
        for src, rel, dst in self.edge_categories['case_similarity']:
            paths.append([(src, rel, dst)])
        return paths
    
    def _generate_subtype_mediated_paths(self):
        """Generate paths that go through subtypes"""
        paths = []
        
        # Case -> subtype -> case (via subtype characterization)
        for case_to_subtype in self.edge_categories['subtype_relations']:
            if case_to_subtype[0] == 'case' and case_to_subtype[2] == 'subtype':
                for subtype_char in self.edge_categories['subtype_relations']:
                    if subtype_char[0] == 'subtype':
                        # Need to find path back to case through molecular data
                        molecular_type = subtype_char[2]
                        for mol_to_case in self.edge_categories['molecular_to_case']:
                            if mol_to_case[0] == molecular_type:
                                path = [case_to_subtype, subtype_char, mol_to_case]
                                paths.append(path)
        
        return paths
    
    def _generate_molecular_mediated_paths(self):
        """Generate paths through molecular data (genes, proteins, CNVs, mutations)"""
        paths = []
        
        # Simple case -> molecular -> case paths
        for case_to_mol in self.edge_categories['case_to_molecular']:
            molecular_type = case_to_mol[2]
            for mol_to_case in self.edge_categories['molecular_to_case']:
                if mol_to_case[0] == molecular_type:
                    path = [case_to_mol, mol_to_case]
                    paths.append(path)
        
        # Case -> molecular -> molecular -> case (with molecular interactions)
        for case_to_mol in self.edge_categories['case_to_molecular']:
            mol_type1 = case_to_mol[2]
            
            # Same molecular type interactions
            for mol_interact in self.edge_categories['molecular_similarity']:
                if mol_interact[0] == mol_type1 and mol_interact[2] == mol_type1:
                    for mol_to_case in self.edge_categories['molecular_to_case']:
                        if mol_to_case[0] == mol_type1:
                            path = [case_to_mol, mol_interact, mol_to_case]
                            paths.append(path)
            
            # Cross-molecular type interactions
            for mol_interact in self.edge_categories['molecular_interactions']:
                if mol_interact[0] == mol_type1:
                    mol_type2 = mol_interact[2]
                    for mol_to_case in self.edge_categories['molecular_to_case']:
                        if mol_to_case[0] == mol_type2:
                            path = [case_to_mol, mol_interact, mol_to_case]
                            paths.append(path)
        
        return paths
    
    def _generate_comprehensive_multihop_paths(self, max_paths_per_length):
        """Generate comprehensive multi-hop paths using DFS exploration"""
        paths = []
        
        # Generate paths of different lengths (3 to max_path_length)
        for target_length in range(3, self.max_path_length + 1):
            length_paths = self._dfs_generate_paths(
                start_node=self.target_node_type,
                current_path=[],
                target_length=target_length,
                max_paths=max_paths_per_length
            )
            paths.extend(length_paths)
            
        return paths
    
    def _dfs_generate_paths(self, start_node, current_path, target_length, max_paths):
        """DFS to generate paths of specific length using all available relations"""
        if len(current_path) == target_length:
            # Check if we end at target node type
            if len(current_path) > 0 and current_path[-1][2] == self.target_node_type:
                return [current_path.copy()]
            else:
                return []
        
        if len(current_path) >= target_length:
            return []
        
        # Determine current node type
        if len(current_path) == 0:
            current_node_type = start_node
        else:
            current_node_type = current_path[-1][2]
        
        all_paths = []
        
        # Explore all possible next steps from current node type
        if current_node_type in self.adjacency_map:
            for rel, next_node_type in self.adjacency_map[current_node_type]:
                # Skip if this would create a path that's too long
                if len(current_path) + 1 > target_length:
                    continue
                
                # Add this edge to the path
                new_step = (current_node_type, rel, next_node_type)
                current_path.append(new_step)
                
                # Recursively explore
                sub_paths = self._dfs_generate_paths(start_node, current_path, target_length, max_paths)
                all_paths.extend(sub_paths)
                
                # Backtrack
                current_path.pop()
                
                # Limit number of paths to prevent explosion
                if len(all_paths) >= max_paths:
                    break
        
        return all_paths[:max_paths]
    
    def _generate_complex_interaction_paths(self):
        """Enhanced complex paths with multiple molecular interactions"""
        paths = []
        
        # Strategy 1: Case -> mol1 -> mol2 -> mol3 -> case (longer molecular chains)
        for case_to_mol1 in self.edge_categories['case_to_molecular']:
            mol_type1 = case_to_mol1[2]
            
            # First molecular interaction
            for mol1_to_mol2 in (self.edge_categories['molecular_similarity'] + 
                               self.edge_categories['molecular_interactions']):
                if mol1_to_mol2[0] == mol_type1:
                    mol_type2 = mol1_to_mol2[2]
                    
                    # Second molecular interaction
                    for mol2_to_mol3 in (self.edge_categories['molecular_similarity'] + 
                                       self.edge_categories['molecular_interactions']):
                        if mol2_to_mol3[0] == mol_type2:
                            mol_type3 = mol2_to_mol3[2]
                            
                            # Back to case
                            for mol3_to_case in self.edge_categories['molecular_to_case']:
                                if mol3_to_case[0] == mol_type3:
                                    path = [case_to_mol1, mol1_to_mol2, mol2_to_mol3, mol3_to_case]
                                    if len(path) <= self.max_path_length:
                                        paths.append(path)
        
        # Strategy 2: Case -> subtype -> molecular -> molecular -> case
        for case_to_subtype in self.edge_categories['subtype_relations']:
            if case_to_subtype[0] == 'case' and case_to_subtype[2] == 'subtype':
                for subtype_to_mol in self.edge_categories['subtype_relations']:
                    if subtype_to_mol[0] == 'subtype':
                        mol_type1 = subtype_to_mol[2]
                        
                        # Molecular interaction
                        for mol_interact in (self.edge_categories['molecular_similarity'] + 
                                           self.edge_categories['molecular_interactions']):
                            if mol_interact[0] == mol_type1:
                                mol_type2 = mol_interact[2]
                                
                                # Back to case
                                for mol_to_case in self.edge_categories['molecular_to_case']:
                                    if mol_to_case[0] == mol_type2:
                                        path = [case_to_subtype, subtype_to_mol, mol_interact, mol_to_case]
                                        if len(path) <= self.max_path_length:
                                            paths.append(path)
        
        # Strategy 3: Case -> similarity -> molecular chain -> case
        for case_sim in self.edge_categories['case_similarity']:
            for case_to_mol in self.edge_categories['case_to_molecular']:
                mol_type1 = case_to_mol[2]
                
                for mol_interact in (self.edge_categories['molecular_similarity'] + 
                                   self.edge_categories['molecular_interactions']):
                    if mol_interact[0] == mol_type1:
                        mol_type2 = mol_interact[2]
                        
                        for mol_to_case in self.edge_categories['molecular_to_case']:
                            if mol_to_case[0] == mol_type2:
                                path = [case_sim, case_to_mol, mol_interact, mol_to_case]
                                if len(path) <= self.max_path_length:
                                    paths.append(path)
        
        return paths
    
    def _generate_hybrid_paths(self):
        """Generate hybrid paths combining different relation types"""
        paths = []
        
        # Strategy 1: Basic hybrid paths (length 3)
        for case_sim in self.edge_categories['case_similarity']:
            for case_to_mol in self.edge_categories['case_to_molecular']:
                mol_type = case_to_mol[2]
                for mol_to_case in self.edge_categories['molecular_to_case']:
                    if mol_to_case[0] == mol_type:
                        # Path: case -> case (similarity) -> molecular -> case
                        path1 = [case_sim, case_to_mol, mol_to_case]
                        if len(path1) <= self.max_path_length:
                            paths.append(path1)
                        
                        # Path: case -> molecular -> case -> case (similarity)
                        path2 = [case_to_mol, mol_to_case, case_sim]
                        if len(path2) <= self.max_path_length:
                            paths.append(path2)
        
        # Strategy 2: Extended hybrid paths (length 4+)
        # Case -> similarity -> subtype -> molecular -> case
        for case_sim in self.edge_categories['case_similarity']:
            for case_to_subtype in self.edge_categories['subtype_relations']:
                if case_to_subtype[0] == 'case' and case_to_subtype[2] == 'subtype':
                    for subtype_to_mol in self.edge_categories['subtype_relations']:
                        if subtype_to_mol[0] == 'subtype':
                            mol_type = subtype_to_mol[2]
                            for mol_to_case in self.edge_categories['molecular_to_case']:
                                if mol_to_case[0] == mol_type:
                                    path = [case_sim, case_to_subtype, subtype_to_mol, mol_to_case]
                                    if len(path) <= self.max_path_length:
                                        paths.append(path)
        
        # Strategy 3: Complex hybrid paths with molecular interactions
        # Case -> molecular -> molecular_interact -> molecular -> case -> similarity
        for case_to_mol1 in self.edge_categories['case_to_molecular']:
            mol_type1 = case_to_mol1[2]
            
            for mol_interact in (self.edge_categories['molecular_similarity'] + 
                               self.edge_categories['molecular_interactions']):
                if mol_interact[0] == mol_type1:
                    mol_type2 = mol_interact[2]
                    
                    for mol2_to_case in self.edge_categories['molecular_to_case']:
                        if mol2_to_case[0] == mol_type2:
                            for case_sim in self.edge_categories['case_similarity']:
                                path = [case_to_mol1, mol_interact, mol2_to_case, case_sim]
                                if len(path) <= self.max_path_length:
                                    paths.append(path)
        
        return paths
    
    def filter_valid_paths(self, paths):
        """Filter paths to ensure they start and end with target node type"""
        valid_paths = []
        for path in paths:
            if len(path) > 0:
                start_node = path[0][0]
                end_node = path[-1][2]
                if start_node == self.target_node_type and end_node == self.target_node_type:
                    valid_paths.append(path)
        return valid_paths
    
    def categorize_metapaths(self, metapaths):
        """Categorize meta-paths for analysis"""
        categories = {
            'direct_similarity': [],
            'subtype_mediated': [],
            'single_molecular': [],
            'molecular_interaction': [],
            'complex_multi_hop': [],
            'hybrid_paths': []
        }
        
        for path in metapaths:
            path_length = len(path)
            
            # Analyze path composition
            has_subtype = any('subtype' in str(step) for step in path)
            has_molecular = any(any(mol in str(step) for mol in ['gene', 'protein', 'cnv', 'mutation']) for step in path)
            has_similarity = any('similar' in str(step) or 'coexpr' in str(step) for step in path)
            molecular_interactions = sum(1 for step in path if step[0] in ['gene', 'protein', 'cnv', 'mutation'] 
                                       and step[2] in ['gene', 'protein', 'cnv', 'mutation'])
            
            # Enhanced categorization
            if path_length == 1 and has_similarity:
                categories['direct_similarity'].append(path)
            elif has_subtype and path_length >= 3:
                categories['subtype_mediated'].append(path)
            elif molecular_interactions >= 1 and path_length >= 3:
                categories['molecular_interaction'].append(path)
            elif has_molecular and path_length == 2:
                categories['single_molecular'].append(path)
            elif path_length >= 4:  # Lowered threshold from >3 to >=4
                categories['complex_multi_hop'].append(path)
            elif has_similarity and has_molecular:
                categories['hybrid_paths'].append(path)
            else:
                categories['single_molecular'].append(path)  # Default category
        
        return categories
    
    def generate_metapath_combinations(self, max_combinations, min_paths_per_set, max_paths_per_set, max_paths_per_length):
        """Generate diverse combinations of meta-paths for comprehensive testing"""
        all_metapaths = self.generate_comprehensive_metapaths(max_paths_per_length)
        valid_metapaths = self.filter_valid_paths(all_metapaths)
        categorized_paths = self.categorize_metapaths(valid_metapaths)
        
        combinations = []
        
        # Strategy 1: Single paths
        if min_paths_per_set <= 1:
            for category, paths in categorized_paths.items():
                for path in paths[:5]:  # Top 5 from each category
                    combinations.append([path])
        
        # Strategy 2: Cross-category combinations for diversity
        category_names = [cat for cat in categorized_paths.keys() if categorized_paths[cat]]
        
        # Generate combinations of different sizes
        for combo_size in range(max(2, min_paths_per_set), max_paths_per_set + 1):
            if combo_size <= len(category_names):
                from itertools import combinations as iter_combinations
                
        
                for cat_combo in list(iter_combinations(category_names, combo_size))[:10]:
                    # Take best path from each selected category
                    combo = []
                    for cat in cat_combo:
                        if categorized_paths[cat]:
                            combo.append(categorized_paths[cat][0])
                    
                    if min_paths_per_set <= len(combo) <= max_paths_per_set:
                        combinations.append(combo)
                
                # Also try combinations within same category
                for category, paths in categorized_paths.items():
                    if len(paths) >= combo_size:
                        for combo in list(iter_combinations(paths, combo_size))[:3]:
                            if min_paths_per_set <= len(combo) <= max_paths_per_set:
                                combinations.append(list(combo))
        
        # Strategy 3: Random diverse combinations (for additional diversity)
        if len(valid_metapaths) >= max_paths_per_set:
            # Select diverse paths by length and category
            selected_paths = []
            
            # Get paths of different lengths
            for length in range(1, self.max_path_length + 1):
                length_paths = [p for p in valid_metapaths if len(p) == length]
                if length_paths:
                    selected_paths.extend(length_paths[:6])
            
            # Generate random combinations if we have enough paths
            if len(selected_paths) >= max_paths_per_set:
                from itertools import combinations as iter_combinations
                for combo_size in range(min_paths_per_set, max_paths_per_set + 1):
                    for combo in list(iter_combinations(selected_paths, combo_size))[:5]:
                        combinations.append(list(combo))
        
        # Remove duplicates and filter by constraints
        filtered_combinations = []
        seen = set()
        
        for combo in combinations:
            if min_paths_per_set <= len(combo) <= max_paths_per_set:
                # Create hashable representation for duplicate detection
                combo_key = tuple(sorted([tuple(path) for path in combo]))
                if combo_key not in seen:
                    seen.add(combo_key)
                    filtered_combinations.append(combo)
        
        # Sort by effectiveness (diversity and path length)
        def combination_score(combo):
            avg_length = sum(len(path) for path in combo) / len(combo)
            diversity = len(set(tuple(path) for path in combo))
            size_bonus = len(combo) * 0.1  # Small bonus for larger combinations
            return diversity / avg_length + size_bonus
        
        filtered_combinations.sort(key=combination_score, reverse=True)
        
        # Return statistics
        category_counts = {cat: len(paths) for cat, paths in categorized_paths.items()}
        
        return filtered_combinations[:max_combinations], len(valid_metapaths), category_counts

def to_numpy(x):
    try:
        import torch
        if isinstance(x, torch.Tensor):
            return x.cpu().numpy()
    except ImportError:
        pass
    return np.array(x)

def build_bipartite_adj(src_idx, dst_idx, shape_src, shape_dst):
    """Build a scipy sparse COO adjacency from src to dst."""
    data = np.ones(len(src_idx), dtype=np.float32)
    return sp.coo_matrix((data, (src_idx, dst_idx)), shape=(shape_src, shape_dst))

def build_meta_path_adj(data, meta_path):
    """Build adjacency among target nodes via sequence of relations in meta_path."""
    src_type, rel_type, dst_type = meta_path[0]
    edge_index = to_numpy(data[src_type, rel_type, dst_type].edge_index)
    src_idx, dst_idx = edge_index
    A = build_bipartite_adj(src_idx, dst_idx, data[src_type].num_nodes, data[dst_type].num_nodes)
    current_dst = dst_type

    for (s_t, r_t, d_t) in meta_path[1:]:
        assert current_dst == s_t, f"Meta-path mismatch: expected {current_dst} but got {s_t}"
        edge_index = to_numpy(data[s_t, r_t, d_t].edge_index)
        s_idx, d_idx = edge_index
        B = build_bipartite_adj(s_idx, d_idx, data[s_t].num_nodes, data[d_t].num_nodes)
        A = A.dot(B)
        current_dst = d_t

    A = A.tocoo()
    if A.shape[0] == A.shape[1]:
        A.setdiag(0)
    A.eliminate_zeros()
    return A

def prepare_data_from_hetero(data, target_node_type='case', meta_paths=None, split_ratios=(0.7, 0.15, 0.15), random_state=MASTER_RANDOM_SEED):
    """From a HeteroData, build fea_list, adj biases, labels and masks for target_node_type."""
    # Features
    if not hasattr(data[target_node_type], 'x'):
        raise ValueError(f"Target node type '{target_node_type}' has no .x features")
    case_feat = to_numpy(data[target_node_type].x)
    N = case_feat.shape[0]
    ft_size = case_feat.shape[1]
    
    # Labels
    if not hasattr(data[target_node_type], 'y'):
        raise ValueError(f"Target node type '{target_node_type}' has no .y labels")
    labels = to_numpy(data[target_node_type].y).astype(int).reshape(-1)
    num_classes = int(labels.max()) + 1
    y_all = np.eye(num_classes, dtype=np.float32)[labels]
    
    # Masks: create if missing
    has_train = hasattr(data[target_node_type], 'train_mask')
    has_val = hasattr(data[target_node_type], 'val_mask')
    has_test = hasattr(data[target_node_type], 'test_mask')
    
    if has_train and has_val and has_test:
        train_mask = to_numpy(getattr(data[target_node_type], 'train_mask')).astype(bool)
        val_mask = to_numpy(getattr(data[target_node_type], 'val_mask')).astype(bool)
        test_mask = to_numpy(getattr(data[target_node_type], 'test_mask')).astype(bool)
    else:
        # Create stratified splits with FIXED random state for reproducibility
        from sklearn.model_selection import StratifiedShuffleSplit
        indices = np.arange(N)
        sss1 = StratifiedShuffleSplit(n_splits=1, test_size=(1 - split_ratios[0]), random_state=random_state)
        train_idx, rest_idx = next(sss1.split(indices, labels))
        
        val_prop = split_ratios[1] / (split_ratios[1] + split_ratios[2])
        sss2 = StratifiedShuffleSplit(n_splits=1, test_size=(1 - val_prop), random_state=random_state)
        val_idx, test_idx = next(sss2.split(rest_idx, labels[rest_idx]))
        val_idx = rest_idx[val_idx]
        test_idx = rest_idx[test_idx]
        
        train_mask = np.zeros(N, dtype=bool)
        val_mask = np.zeros(N, dtype=bool)
        test_mask = np.zeros(N, dtype=bool)
        train_mask[train_idx] = True
        val_mask[val_idx] = True
        test_mask[test_idx] = True
        
    # Adjacencies
    if meta_paths is None:
        raise ValueError("Please provide a list of meta-paths for adjacency")
    
    adj_list = []
    for meta_path in meta_paths:
        if len(meta_path) == 0:
            raise ValueError("Meta-path cannot be empty")
        first = meta_path[0]
        last = meta_path[-1]
        if first[0] != target_node_type or last[2] != target_node_type:
            raise ValueError(f"Meta-path {meta_path} must start and end with '{target_node_type}'")
        A = build_meta_path_adj(data, meta_path)
        A_mat = A.toarray() if sp.isspmatrix(A) else np.array(A)
        adj_list.append(A_mat)
        
    # Feature list
    fea_list = [case_feat for _ in adj_list]
    
    # Batch dim
    fea_list = [f[np.newaxis] for f in fea_list]
    y_train = np.zeros_like(y_all); y_train[train_mask] = y_all[train_mask]
    y_val = np.zeros_like(y_all); y_val[val_mask] = y_all[val_mask]
    y_test = np.zeros_like(y_all); y_test[test_mask] = y_all[test_mask]
    y_train = y_train[np.newaxis]
    y_val = y_val[np.newaxis]
    y_test = y_test[np.newaxis]
    train_mask_b = train_mask[np.newaxis].astype(np.int32)
    val_mask_b = val_mask[np.newaxis].astype(np.int32)
    test_mask_b = test_mask[np.newaxis].astype(np.int32)
    
    # Biases via process.adj_to_bias
    biases_list = []
    for A_mat in adj_list:
        bias = process.adj_to_bias(np.expand_dims(A_mat, 0), [N], nhood=1)
        if bias.ndim == 2:
            bias = bias[np.newaxis]
        biases_list.append(bias)
    
    return fea_list, biases_list, y_train, y_val, y_test, train_mask_b, val_mask_b, test_mask_b, N, ft_size, num_classes, y_all


def train_model_with_metapaths(fea_list, biases_list, y_train, y_val, y_test, 
                              train_mask, val_mask, test_mask, nb_nodes, ft_size, nb_classes,
                              batch_size, nb_epochs, patience, lr, l2_coef, hid_units, n_heads,
                              residual, nonlinearity, model, logger, metapath_idx):
    """Train model with specific meta-paths - ensuring independence between experiments"""
    
    checkpt_file = f'HAN_Model/pre_trained/hetero_case_metapath_{metapath_idx}.ckpt'
    
    # Set deterministic but different seeds for each experiment
    experiment_seed = MASTER_RANDOM_SEED + (metapath_idx * EXPERIMENT_OFFSET)
    
    # Set ALL random seeds BEFORE creating the graph
    random.seed(experiment_seed)
    np.random.seed(experiment_seed)
    
    # Reset TensorFlow state
    tf.reset_default_graph()
    
    # Set TensorFlow seed after resetting the graph
    tf.set_random_seed(experiment_seed)
    
    logger.logger.debug(f"Using random seed: {experiment_seed} for meta-path set {metapath_idx}")
    
    # Disable TF v2 behavior and configure GPU
    tf.disable_v2_behavior()
    config = tf.ConfigProto()
    config.gpu_options.allow_growth = True
    config.allow_soft_placement = True
    
    # Suppress TensorFlow session logging
    config.log_device_placement = False
    
    # Redirect stdout and stderr to suppress all unwanted output
    import sys
    from contextlib import redirect_stdout, redirect_stderr
    import io
    
    with tf.Graph().as_default():
        # Set TensorFlow seed within the graph context
        tf.set_random_seed(experiment_seed)
        
        with tf.name_scope('input'):
            ftr_in_list = [tf.placeholder(dtype=tf.float32, shape=(batch_size, nb_nodes, ft_size), name=f'ftr_in_{i}')
                           for i in range(len(fea_list))]
            bias_in_list = [tf.placeholder(dtype=tf.float32, shape=(batch_size, nb_nodes, nb_nodes), name=f'bias_in_{i}')
                            for i in range(len(biases_list))]
            lbl_in = tf.placeholder(dtype=tf.int32, shape=(batch_size, nb_nodes, nb_classes), name='lbl_in')
            msk_in = tf.placeholder(dtype=tf.int32, shape=(batch_size, nb_nodes), name='msk_in')
            attn_drop = tf.placeholder(dtype=tf.float32, shape=(), name='attn_drop')
            ffd_drop = tf.placeholder(dtype=tf.float32, shape=(), name='ffd_drop')
            is_train = tf.placeholder(dtype=tf.bool, shape=(), name='is_train')
        
        # Suppress output during model inference creation
        with redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
            logits, final_embedding, att_val = model.inference(ftr_in_list, nb_classes, nb_nodes, is_train,
                                                               attn_drop, ffd_drop,
                                                               bias_mat_list=bias_in_list,
                                                               hid_units=hid_units, n_heads=n_heads,
                                                               residual=residual, activation=nonlinearity)
        
        log_resh = tf.reshape(logits, [-1, nb_classes])
        lab_resh = tf.reshape(lbl_in, [-1, nb_classes])
        msk_resh = tf.reshape(msk_in, [-1])
        
        loss = model.masked_softmax_cross_entropy(log_resh, lab_resh, msk_resh)
        accuracy = model.masked_accuracy(log_resh, lab_resh, msk_resh)
        train_op = model.training(loss, lr, l2_coef)
        
        saver = tf.train.Saver()
        init_op = tf.group(tf.global_variables_initializer(), tf.local_variables_initializer())
        
        # Suppress TensorFlow logging during session
        old_log_level = tf.logging.get_verbosity()
        tf.logging.set_verbosity(tf.logging.ERROR)
        
        with tf.Session(config=config) as sess:
            # Suppress output during session initialization and training
            with redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
                sess.run(init_op)
                
                vlss_mn = np.inf
                vacc_mx = 0.0
                curr_step = 0
                
                for epoch in range(nb_epochs):
                    # Training
                    fd = {ph: arr for ph, arr in zip(ftr_in_list, fea_list)}
                    fd.update({ph: arr for ph, arr in zip(bias_in_list, biases_list)})
                    fd.update({lbl_in: y_train, msk_in: train_mask, is_train: True, attn_drop: 0.6, ffd_drop: 0.6})
                    
                    _, loss_value_tr, acc_tr, att_val_train = sess.run([train_op, loss, accuracy, att_val], feed_dict=fd)
                    
                    # Validation
                    fd_val = {ph: arr for ph, arr in zip(ftr_in_list, fea_list)}
                    fd_val.update({ph: arr for ph, arr in zip(bias_in_list, biases_list)})
                    fd_val.update({lbl_in: y_val, msk_in: val_mask, is_train: False, attn_drop: 0.0, ffd_drop: 0.0})
                    
                    loss_value_vl, acc_vl = sess.run([loss, accuracy], feed_dict=fd_val)
                    
                    # Only allow our logger output - no suppression here
                    logger.log_training_progress(epoch, loss_value_tr, acc_tr, loss_value_vl, acc_vl)
                    
                    # Early stopping logic
                    if acc_vl >= vacc_mx or loss_value_vl <= vlss_mn:
                        if acc_vl >= vacc_mx and loss_value_vl <= vlss_mn:
                            saver.save(sess, checkpt_file)
                        vacc_mx = max(acc_vl, vacc_mx)
                        vlss_mn = min(loss_value_vl, vlss_mn)
                        curr_step = 0
                    else:
                        curr_step += 1
                        if curr_step == patience:
                            break
                
                # Test evaluation - suppress checkpoint restoration message
                saver.restore(sess, checkpt_file)
                fd_test = {ph: arr for ph, arr in zip(ftr_in_list, fea_list)}
                fd_test.update({ph: arr for ph, arr in zip(bias_in_list, biases_list)})
                fd_test.update({lbl_in: y_test, msk_in: test_mask, is_train: False, attn_drop: 0.0, ffd_drop: 0.0})
                
                loss_value_ts, acc_ts = sess.run([loss, accuracy], feed_dict=fd_test)
        
        # Restore original log level
        tf.logging.set_verbosity(old_log_level)
        
        return acc_tr, vacc_mx, acc_ts


def train_and_save_final_model(fea_list, biases_list, y_train, y_val, y_test, 
                              train_mask, val_mask, test_mask, nb_nodes, ft_size, nb_classes,
                              metapath_set, model_save_path, nb_epochs, patience,
                              lr=0.005, l2_coef=0.001, hid_units=[8], n_heads=[8, 1],
                              residual=False, nonlinearity=tf.nn.elu, model=HeteGAT_multi,
                              verbose=True):
    
    if verbose:
        print("\n" + "="*80)
        print("TRAINING FINAL MODEL WITH SELECTED METAPATH SET")
        print("="*80)
        print(f"\nModel will be saved to: {model_save_path}")
        print(f"Training configuration:")
        print(f"  - Epochs: {nb_epochs}")
        print(f"  - Patience: {patience}")
        print(f"  - Learning rate: {lr}")
        print(f"  - L2 coefficient: {l2_coef}")
        print(f"  - Hidden units: {hid_units}")
        print(f"  - Attention heads: {n_heads}")
        
        print(f"\nSelected metapath set ({len(metapath_set)} paths):")
        for i, metapath in enumerate(metapath_set):
            path_str = " -> ".join([f"{src}--{rel}-->{dst}" for src, rel, dst in metapath])
            print(f"  {i+1}. {path_str}")
    
    # Create directory for model if it doesn't exist
    os.makedirs(os.path.dirname(model_save_path), exist_ok=True)
    
    # Set fixed seed for final training (use same global seed for consistency)
    random.seed(MASTER_RANDOM_SEED)
    np.random.seed(MASTER_RANDOM_SEED)
    
    # Reset TensorFlow state
    tf.reset_default_graph()
    tf.set_random_seed(MASTER_RANDOM_SEED)
    
    if verbose:
        print(f"\nUsing master random seed: {MASTER_RANDOM_SEED} for final training")
        print("\nStarting training...")    
    
    # Configure TensorFlow
    tf.disable_v2_behavior()
    config = tf.ConfigProto()
    config.gpu_options.allow_growth = True
    config.allow_soft_placement = True
    config.log_device_placement = False
    
    batch_size = 1
    start_time = time.time()
    
    with tf.Graph().as_default():
        tf.set_random_seed(MASTER_RANDOM_SEED)
        
        with tf.name_scope('input'):
            ftr_in_list = [tf.placeholder(dtype=tf.float32, shape=(batch_size, nb_nodes, ft_size), name=f'ftr_in_{i}')
                           for i in range(len(fea_list))]
            bias_in_list = [tf.placeholder(dtype=tf.float32, shape=(batch_size, nb_nodes, nb_nodes), name=f'bias_in_{i}')
                            for i in range(len(biases_list))]
            lbl_in = tf.placeholder(dtype=tf.int32, shape=(batch_size, nb_nodes, nb_classes), name='lbl_in')
            msk_in = tf.placeholder(dtype=tf.int32, shape=(batch_size, nb_nodes), name='msk_in')
            attn_drop = tf.placeholder(dtype=tf.float32, shape=(), name='attn_drop')
            ffd_drop = tf.placeholder(dtype=tf.float32, shape=(), name='ffd_drop')
            is_train = tf.placeholder(dtype=tf.bool, shape=(), name='is_train')
        
        # Create model
        logits, final_embedding, att_val = model.inference(ftr_in_list, nb_classes, nb_nodes, is_train,
                                                           attn_drop, ffd_drop,
                                                           bias_mat_list=bias_in_list,
                                                           hid_units=hid_units, n_heads=n_heads,
                                                           residual=residual, activation=nonlinearity)
        
        log_resh = tf.reshape(logits, [-1, nb_classes])
        lab_resh = tf.reshape(lbl_in, [-1, nb_classes])
        msk_resh = tf.reshape(msk_in, [-1])
        
        loss = model.masked_softmax_cross_entropy(log_resh, lab_resh, msk_resh)
        accuracy = model.masked_accuracy(log_resh, lab_resh, msk_resh)
        train_op = model.training(loss, lr, l2_coef)
        
        saver = tf.train.Saver()
        init_op = tf.group(tf.global_variables_initializer(), tf.local_variables_initializer())
        
        with tf.Session(config=config) as sess:
            sess.run(init_op)
            
            vlss_mn = np.inf
            vacc_mx = 0.0
            curr_step = 0
            best_epoch = 0
            
            for epoch in range(nb_epochs):
                # Training
                fd = {ph: arr for ph, arr in zip(ftr_in_list, fea_list)}
                fd.update({ph: arr for ph, arr in zip(bias_in_list, biases_list)})
                fd.update({lbl_in: y_train, msk_in: train_mask, is_train: True, attn_drop: 0.6, ffd_drop: 0.6})
                
                _, loss_value_tr, acc_tr, att_val_train = sess.run([train_op, loss, accuracy, att_val], feed_dict=fd)
                
                # Validation
                fd_val = {ph: arr for ph, arr in zip(ftr_in_list, fea_list)}
                fd_val.update({ph: arr for ph, arr in zip(bias_in_list, biases_list)})
                fd_val.update({lbl_in: y_val, msk_in: val_mask, is_train: False, attn_drop: 0.0, ffd_drop: 0.0})
                
                loss_value_vl, acc_vl = sess.run([loss, accuracy], feed_dict=fd_val)
                
                # Progress logging
                if verbose and (epoch % 10 == 0 or epoch < 10):
                    print(f"Epoch {epoch:3d}: Train Acc: {acc_tr:.4f}, Val Acc: {acc_vl:.4f}, Train Loss: {loss_value_tr:.4f}, Val Loss: {loss_value_vl:.4f}")
                
                # Early stopping and model saving logic
                if acc_vl >= vacc_mx or loss_value_vl <= vlss_mn:
                    if acc_vl >= vacc_mx and loss_value_vl <= vlss_mn:
                        # Save the best model
                        saver.save(sess, model_save_path)
                        best_epoch = epoch
                        if verbose:
                            print(f"  -> Saved new best model at epoch {epoch}")
                    vacc_mx = max(acc_vl, vacc_mx)
                    vlss_mn = min(loss_value_vl, vlss_mn)
                    curr_step = 0
                else:
                    curr_step += 1
                    if curr_step == patience:
                        if verbose:
                            print(f"Early stopping at epoch {epoch}")
                        break
            
            # Load best model and evaluate on test set
            saver.restore(sess, model_save_path)
            fd_test = {ph: arr for ph, arr in zip(ftr_in_list, fea_list)}
            fd_test.update({ph: arr for ph, arr in zip(bias_in_list, biases_list)})
            fd_test.update({lbl_in: y_test, msk_in: test_mask, is_train: False, attn_drop: 0.0, ffd_drop: 0.0})
            
            loss_value_ts, acc_ts = sess.run([loss, accuracy], feed_dict=fd_test)
    
    training_time = time.time() - start_time
    
    if verbose:
        print(f"\n" + "="*60)
        print("FINAL TRAINING COMPLETED")
        print("="*60)
        print(f"Training time: {training_time:.2f} seconds")
        print(f"Best epoch: {best_epoch}")
        print(f"Final train accuracy: {acc_tr:.4f}")
        print(f"Best validation accuracy: {vacc_mx:.4f}")
        print(f"Final test accuracy: {acc_ts:.4f}")
        print(f"Model saved to: {model_save_path}")
    
    return {
        'train_accuracy': float(acc_tr),
        'val_accuracy': float(vacc_mx),
        'test_accuracy': float(acc_ts),
        'training_time': training_time,
        'best_epoch': best_epoch,
        'model_path': model_save_path,
        'metapath_set': metapath_set
    }


def load_metapath_from_csv(csv_file_path, rank_number):

    try:
        df = pd.read_csv(csv_file_path)
    except FileNotFoundError:
        raise ValueError(f"\nCSV file not found: {csv_file_path}")
    
    if rank_number < 1 or rank_number > len(df):
        raise ValueError(f"\nRank number {rank_number} is out of range. Available ranks: 1-{len(df)}")
    
    # Get the row (convert to 0-indexed)
    row = df.iloc[rank_number - 1]
    metapath_description = row['metapath_description']
    
    print(f"\nLoading metapath set from rank {rank_number}:")
    print(f"  \nTest accuracy: {row['test_accuracy']:.4f}")
    print(f"  \nDescription: {metapath_description}")
    
    # Parse the metapath description back into metapath structure
    metapath_set = parse_metapath_description(metapath_description)
    
    # Debug: Print parsed result
    print(f" \n Parsed {len(metapath_set)} metapaths:")
    for i, metapath in enumerate(metapath_set):
        print(f"    {i+1}. {metapath}")
    
    if len(metapath_set) == 0:
        print("  \nWARNING: No metapaths were successfully parsed!")
        print("  This will cause training to fail.")
        
        # Try to provide helpful debugging info
        print("  Attempting to debug the parsing...")
        paths = metapath_description.split(" | ")
        print(f"  Split into {len(paths)} path strings:")
        for i, path in enumerate(paths):
            print(f"    Path {i+1}: '{path}'")
            if " -> " in path:
                edges = path.split(" -> ")
                print(f"      Split into {len(edges)} edges:")
                for j, edge in enumerate(edges):
                    print(f"        Edge {j+1}: '{edge}'")
    
    return metapath_set, row


def parse_metapath_description(description):

    metapath_set = []
    
    # Split by " | " to get individual paths
    path_strings = description.split(" | ")
    
    for path_string in path_strings:
        path_string = path_string.strip()
        
        # Check if this is a simple edge (src--rel-->dst) or complex path
        if " -> " in path_string:
            # Complex path with multiple hops
            metapath = []
            edges = path_string.split(" -> ")
            
            for edge in edges:
                edge = edge.strip()
                # Parse "src--rel-->dst" format
                if "--" in edge and "-->" in edge:
                    # Find the split point between src and relation
                    first_dash_pos = edge.find("--")
                    last_arrow_pos = edge.rfind("-->")
                    
                    if first_dash_pos != -1 and last_arrow_pos != -1 and first_dash_pos < last_arrow_pos:
                        src = edge[:first_dash_pos]
                        rel = edge[first_dash_pos+2:last_arrow_pos]
                        dst = edge[last_arrow_pos+3:]
                        metapath.append((src, rel, dst))
            
            if metapath:  # Only add non-empty metapaths
                metapath_set.append(metapath)
        else:
            # Simple single edge path
            edge = path_string.strip()
            if "--" in edge and "-->" in edge:
                # Find the split point between src and relation
                first_dash_pos = edge.find("--")
                last_arrow_pos = edge.rfind("-->")
                
                if first_dash_pos != -1 and last_arrow_pos != -1 and first_dash_pos < last_arrow_pos:
                    src = edge[:first_dash_pos]
                    rel = edge[first_dash_pos+2:last_arrow_pos]
                    dst = edge[last_arrow_pos+3:]
                    metapath_set.append([(src, rel, dst)])
    
    return metapath_set


def train_final_model_from_csv(hetero_data, csv_file_path, rank_number, model_save_path,
                              target_node_type='case', nb_epochs=200, patience=100,
                              split_ratios=(0.7, 0.15, 0.15)):

    
    print("\n" + "="*80)
    print("TRAINING FINAL MODEL FROM CSV RESULTS")
    print("="*80)
    
    # Load metapath set from CSV
    metapath_set, row_info = load_metapath_from_csv(csv_file_path, rank_number)
    
    # Validate that metapaths were successfully parsed
    if len(metapath_set) == 0:
        print("\n" + "="*80)
        print("ERROR: No metapaths were successfully parsed!")
        print("="*80)
        print("This indicates an issue with the metapath description format in the CSV.")
        print("Please check the CSV file and the metapath description format.")
        print("Expected format: 'src--rel-->dst | src--rel-->dst -> dst--rel2-->dst2'")
        print("\nTraining cannot proceed without valid metapaths.")
        return None
    
    print(f"\nLoaded metapath set from rank {rank_number}:")
    print(f"  Original test accuracy: {row_info['test_accuracy']:.4f}")
    print(f"  Original val accuracy: {row_info['val_accuracy']:.4f}")
    print(f"  Number of metapaths: {len(metapath_set)}")
    
    # Prepare data with the selected metapath set
    print(f"\nPreparing data with {len(metapath_set)} metapaths...")
    try:
        fea_list, biases_list, y_train, y_val, y_test, train_mask, val_mask, test_mask, nb_nodes, ft_size, nb_classes, y_all = \
            prepare_data_from_hetero(hetero_data, target_node_type=target_node_type, 
                                   meta_paths=metapath_set, split_ratios=split_ratios)
    except Exception as e:
        print(f"\nERROR in data preparation: {str(e)}")
        print("This might indicate invalid metapath format.")
        return None
    
    print(f"Data preparation complete:")
    print(f"  Number of nodes: {nb_nodes}")
    print(f"  Feature size: {ft_size}")
    print(f"  Number of classes: {nb_classes}")
    print(f"  Train samples: {train_mask.sum()}")
    print(f"  Val samples: {val_mask.sum()}")
    print(f"  Test samples: {test_mask.sum()}")
    
    # Train and save the final model
    training_results = train_and_save_final_model(
        fea_list=fea_list,
        biases_list=biases_list,
        y_train=y_train,
        y_val=y_val,
        y_test=y_test,
        train_mask=train_mask,
        val_mask=val_mask,
        test_mask=test_mask,
        nb_nodes=nb_nodes,
        ft_size=ft_size,
        nb_classes=nb_classes,
        metapath_set=metapath_set,
        model_save_path=model_save_path,
        nb_epochs=nb_epochs,
        patience=patience,
    )
    
    # Add original experiment info to results
    training_results.update({
        'original_rank': rank_number,
        'original_test_accuracy': float(row_info['test_accuracy']),
        'original_val_accuracy': float(row_info['val_accuracy']),
        'csv_source': csv_file_path
    })
    
    print(f"\n" + "="*80)
    print("FINAL MODEL TRAINING SUMMARY")
    print("="*80)
    print(f"Selected metapath rank: {rank_number}")
    print(f"Original accuracy: {row_info['test_accuracy']:.4f}")
    print(f"Final model accuracy: {training_results['test_accuracy']:.4f}")
    print(f"Model saved to: {model_save_path}")
    
    return training_results


def get_available_metapath_rankings(csv_file_path, top_n=10):

    try:
        df = pd.read_csv(csv_file_path)
    except FileNotFoundError:
        print(f"CSV file not found: {csv_file_path}")
        return None
    
    print(f"\nTop {min(top_n, len(df))} metapath sets from {csv_file_path}:")
    print("="*120)
    print(f"{'Rank':<6} {'Test Acc':<10} {'Val Acc':<10} {'Train Acc':<10} {'#Paths':<8} {'Description':<70}")
    print("="*120)
    
    for i, row in df.head(top_n).iterrows():
        rank = row.get('rank', i+1)
        desc = row['metapath_description']
        if len(desc) > 65:
            desc = desc[:62] + "..."
        
        print(f"{rank:<6} {row['test_accuracy']:<10.4f} {row['val_accuracy']:<10.4f} "
              f"{row['train_accuracy']:<10.4f} {row['num_metapaths']:<8} {desc:<70}")
    
    print("="*120)
    print(f"Total available ranks: 1-{len(df)}")


# Main function with enhanced meta-path generation
def main(hetero_data, target_node_type, max_path_length, max_paths_per_length, max_metapath_combinations, min_paths_per_set, max_paths_per_set):
    """Main function with enhanced meta-path generation and comprehensive logging"""
    
    # Initialize logger
    logger = MetaPathLogger()
    logger.log_experiment_start(hetero_data)
    
    # Generate meta-paths with enhanced generator
    generator = EnhancedMetaPathGenerator(hetero_data, target_node_type, max_path_length)
    metapath_combinations, total_paths, category_counts = generator.generate_metapath_combinations(
        max_metapath_combinations, min_paths_per_set, max_paths_per_set, max_paths_per_length
    ) 
    
    logger.log_metapath_generation(total_paths, category_counts)
    print(f"\nGenerated {len(metapath_combinations)} meta-path combinations")
    print(f"\nPath set sizes range from {min_paths_per_set} to {max_paths_per_set} paths")
    
    # Log distribution of combination sizes
    size_distribution = {}
    for combo in metapath_combinations:
        size = len(combo)
        size_distribution[size] = size_distribution.get(size, 0) + 1
    
    print("\nMeta-path combination size distribution:")
    for size in sorted(size_distribution.keys()):
        print(f"  {size} paths: {size_distribution[size]} combinations")
    
    # Training configuration
    batch_size = 1
    nb_epochs = 100
    patience = 50
    lr = 0.005
    l2_coef = 0.001
    hid_units = [8]
    n_heads = [8, 1]
    residual = False
    nonlinearity = tf.nn.elu
    model = HeteGAT_multi
    
    # Test each meta-path combination
    for metapath_idx, meta_paths in enumerate(metapath_combinations):
        logger.log_metapath_set_start(metapath_idx, meta_paths)
        
        start_time = time.time()
        
        try:
            # Ensure clean slate for each experiment
            tf.reset_default_graph()
            
            # Prepare data with current meta-paths
            fea_list, biases_list, y_train, y_val, y_test, train_mask, val_mask, test_mask, nb_nodes, ft_size, nb_classes, y_all = \
                prepare_data_from_hetero(hetero_data, target_node_type= target_node_type, meta_paths=meta_paths)
            
            # Train model
            train_acc, val_acc, test_acc = train_model_with_metapaths(
                fea_list, biases_list, y_train, y_val, y_test, 
                train_mask, val_mask, test_mask, nb_nodes, ft_size, nb_classes,
                batch_size, nb_epochs, patience, lr, l2_coef, hid_units, n_heads,
                residual, nonlinearity, model, logger, metapath_idx
            )
            
            training_time = time.time() - start_time
            logger.log_metapath_result(meta_paths, train_acc, val_acc, test_acc, training_time)
            
        except Exception as e:
            logger.logger.error(f"Error with meta-path set {metapath_idx}: {str(e)}")
            continue
    
    # Final summary
    logger.log_final_summary()
    
    # Return results for programmatic access, but don't print them
    return {
        'best_metapath': logger.best_metapath,
        'best_accuracy': logger.best_accuracy,
        'all_results': logger.results,
        'total_experiments': len(logger.results),
        'experiment_completed': True
    }

def run_comprehensive_metapath_experiment(hetero_data, target_node_type='case', max_path_length=4, max_paths_per_length=20, max_metapath_combinations=10, min_paths_per_set=5, max_paths_per_set=6):
    """Run the comprehensive metapath experiment"""
    print("Starting comprehensive metapath experiment...")
    print("\nThis will test multiple metapath combinations including all relations.\n")
    
    # Run the main experiment
    results = main(
        hetero_data,
        target_node_type,
        max_path_length,
        max_paths_per_length,
        max_metapath_combinations,
        min_paths_per_set,
        max_paths_per_set
    )
    
    return results